# Этап 4. Анализ и оптимизация
## Банковский консультант: RAG-система

1. Оценка качества RAG через Ragas (faithfulness, answer_relevancy, context_utilization)
2. Оптимизация производительности: кеширование, батчинг, сравнение до/после

## 0. Инициализация

In [ ]:
import os
import json
import time
import pandas as pd
from dotenv import load_dotenv

load_dotenv()

with open("rag_config.json", encoding="utf-8") as f:
    config = json.load(f)

CONNECTION_STRING = config["connection_string"]
COLLECTION_NAME   = config["collection_name"]

OPENROUTER_API_KEY  = os.getenv("OPENROUTER_API_KEY", "your-openrouter-api-key")
OPENROUTER_BASE_URL = "https://openrouter.ai/api/v1"

print(json.dumps(config, ensure_ascii=False, indent=2))

In [3]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import PGVector
from langchain_openai import ChatOpenAI
from langchain.prompts import ChatPromptTemplate, MessagesPlaceholder

embeddings_model = HuggingFaceEmbeddings(
    model_name=config["embedding_model"],
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True, "batch_size": 32},
)

vectorstore = PGVector(
    collection_name=COLLECTION_NAME,
    connection_string=CONNECTION_STRING,
    embedding_function=embeddings_model,
)

llm = ChatOpenAI(
    model="mistralai/mixtral-8x7b-instruct",
    openai_api_key=OPENROUTER_API_KEY,
    openai_api_base=OPENROUTER_BASE_URL,
    temperature=0.1,
    max_tokens=1024,
)

base_retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 4},
)

In [4]:
# Восстанавливаем RAG-цепочку из Этапа 3
from langchain.schema import Document
from typing import List

SYSTEM_PROMPT = (
    "Вы -- виртуальный консультант банка 'Альфа-Финанс'.\n"
    "Ваша задача -- давать точные ответы на вопросы клиентов о банковских продуктах.\n"
    "\n"
    "ПРАВИЛА:\n"
    "1. Отвечайте ТОЛЬКО на основе предоставленного контекста.\n"
    "2. Если информация отсутствует -- скажите об этом прямо.\n"
    "3. Цитируйте источник: [Источник: название документа].\n"
    "4. Приводите конкретные цифры из документов.\n"
    "\n"
    "КОНТЕКСТ ИЗ ДОКУМЕНТОВ БАНКА:\n"
    "{context}"
)

CHAT_PROMPT = ChatPromptTemplate.from_messages([
    ("system", SYSTEM_PROMPT),
    MessagesPlaceholder(variable_name="chat_history"),
    ("human", "{question}"),
])


def format_context(docs: List[Document]):
    if not docs:
        return "Информация не найдена.", []
    parts, sources = [], []
    for i, doc in enumerate(docs, 1):
        title = doc.metadata.get("title", "Документ банка")
        parts.append(f"[{i}] {title}:\n{doc.page_content}")
        if title not in sources:
            sources.append(title)
    return "\n\n".join(parts), sources


def rag_answer(question: str) -> dict:
    """Получает ответ и возвращает словарь для Ragas."""
    docs             = base_retriever.invoke(question)
    context, sources = format_context(docs)

    messages = CHAT_PROMPT.format_messages(
        context=context,
        chat_history=[],
        question=question,
    )
    response = llm.invoke(messages)
    answer   = response.content

    if sources and "[Источник:" not in answer:
        answer += f"\n\n[Источник: {', '.join(sources)}]"

    return {
        "question":  question,
        "answer":    answer,
        "contexts":  [doc.page_content for doc in docs],
        "sources":   sources,
    }

---
## 1. Оценка качества через Ragas

**Метрики Ragas:**
- **Faithfulness** -- насколько ответ фактически соответствует контексту (нет галлюцинаций)
- **Answer Relevancy** -- насколько ответ релевантен заданному вопросу
- **Context Utilization** -- насколько извлечённый контекст релевантен вопросу (качество ретрива)

Все метрики в диапазоне [0, 1], выше -- лучше.

In [ ]:
# Тестовый датасет из 10 вопросов для Ragas
EVAL_QUESTIONS = [
    "Какая минимальная сумма потребительского кредита?",
    "Какой максимальный срок ипотеки в банке?",
    "Что такое семейная ипотека и какова её ставка?",
    "Можно ли досрочно погасить кредит без штрафа?",
    "Какие ставки по вкладу Максимальный доход?",
    "Какой минимальный первоначальный взнос по ипотеке?",
    "Какие требования к возрасту заёмщика?",
    "Можно ли использовать материнский капитал при ипотеке?",
    "Какие способы погашения кредита доступны?",
    "Что нужно для открытия вклада с максимальной ставкой?",
]

print(f"Тестовый датасет: {len(EVAL_QUESTIONS)} вопросов")
print("Запускаем ретрив и генерацию ответов...")

eval_data = []
for i, q in enumerate(EVAL_QUESTIONS, 1):
    print(f"  [{i}/{len(EVAL_QUESTIONS)}] {q[:60]}...")
    t0     = time.time()
    result = rag_answer(q)
    elapsed = time.time() - t0
    result["latency"] = elapsed
    eval_data.append(result)

print(f"\nГотово. Среднее время ответа: {sum(r['latency'] for r in eval_data)/len(eval_data):.2f}с")

In [ ]:
from datasets import Dataset
from ragas import evaluate
from ragas.metrics import (
    faithfulness,
    answer_relevancy,
)
from ragas.metrics import ContextUtilization
from langchain_openai import ChatOpenAI as RagasChatOpenAI, OpenAIEmbeddings

context_utilization = ContextUtilization()

# Подготовка датасета для Ragas (v0.2 column names)
ragas_data = {
    "user_input":          [d["question"]  for d in eval_data],
    "response":            [d["answer"]    for d in eval_data],
    "retrieved_contexts":  [d["contexts"]  for d in eval_data],
}

ragas_dataset = Dataset.from_dict(ragas_data)

# Ragas требует LLM и эмбеддинги совместимые с OpenAI API
# Используем тот же OpenRouter
ragas_llm = RagasChatOpenAI(
    model="mistralai/mixtral-8x7b-instruct",
    openai_api_key=OPENROUTER_API_KEY,
    openai_api_base=OPENROUTER_BASE_URL,
    temperature=0,
)

# Для эмбеддингов Ragas используем text-embedding-ada-002 через OpenRouter
# (или можно обернуть наши HuggingFace эмбеддинги)
ragas_embeddings = OpenAIEmbeddings(
    model="openai/text-embedding-ada-002",
    openai_api_key=OPENROUTER_API_KEY,
    openai_api_base=OPENROUTER_BASE_URL,
)

print("Датасет для Ragas подготовлен:")
print(f"  Записей: {len(ragas_dataset)}")
print(f"  Колонки: {ragas_dataset.column_names}")

In [ ]:
# Запуск Ragas evaluation
print("Запуск Ragas evaluation...")
print("(это может занять 3-5 минут из-за LLM-вызовов для каждого примера)")
print()

t0 = time.time()

ragas_result = evaluate(
    dataset=ragas_dataset,
    metrics=[faithfulness, answer_relevancy, context_utilization],
    llm=ragas_llm,
    embeddings=ragas_embeddings,
    raise_exceptions=False,
)

ragas_time = time.time() - t0
print(f"\nEvaluation завершён за {ragas_time:.1f}с")

In [ ]:
# Итоговые метрики
print("РЕЗУЛЬТАТЫ RAGAS EVALUATION")

metrics_map = {
    "faithfulness":       "Faithfulness    (нет галлюцинаций)",
    "answer_relevancy":   "Answer Relevancy (ответ к вопросу)",
    "context_utilization":  "Context Utilization (контекст к вопросу)",
}

ragas_scores = {}
score_summary = dict(ragas_result._repr_dict)
for key, label in metrics_map.items():
    score = score_summary.get(key, None)
    if score is not None:
        ragas_scores[key] = round(float(score), 4)
        icon = "GOOD" if score >= 0.8 else ("MED" if score >= 0.6 else "LOW")
        print(f"  {label:<42} : {score:.4f}  [{icon}]")
    else:
        print(f"  {label:<42} : N/A")

overall = sum(ragas_scores.values()) / len(ragas_scores) if ragas_scores else 0
print(f"  {'Среднее по всем метрикам':<42} : {overall:.4f}")
print()
print("Интерпретация:")
print("  >= 0.8 -- отличное качество")
print("  >= 0.6 -- приемлемое качество")
print("  <  0.6 -- требует улучшения")

In [ ]:
# Детализация по вопросам
ragas_df = ragas_result.to_pandas()
print("\nДетализация по вопросам:")
print("-" * 100)

display_cols = ["user_input", "faithfulness", "answer_relevancy", "context_utilization"]
available_cols = [c for c in display_cols if c in ragas_df.columns]

pd.set_option("display.max_colwidth", 50)
pd.set_option("display.float_format", "{:.3f}".format)
print(ragas_df[available_cols].to_string(index=False))

# Анализ слабых мест
print("\nАнализ: вопросы с наименьшим faithfulness:")
if "faithfulness" in ragas_df.columns:
    weak = ragas_df.nsmallest(3, "faithfulness")[["user_input", "faithfulness"]]
    for _, row in weak.iterrows():
        print(f"  {row['faithfulness']:.3f} | {row['user_input'][:70]}")

In [ ]:
# Выводы по метрикам Ragas
print("АНАЛИЗ РЕЗУЛЬТАТОВ И РЕКОМЕНДАЦИИ")

analysis = {
    "faithfulness": {
        "name": "Faithfulness",
        "desc": "Доля утверждений в ответе, подтверждённых контекстом",
        "improvement": [
            "Снизить temperature LLM до 0.0",
            "Добавить в промпт явный запрет отвечать вне контекста",
            "Использовать Self-Query для более точного ретрива",
        ]
    },
    "answer_relevancy": {
        "name": "Answer Relevancy",
        "desc": "Насколько ответ релевантен вопросу",
        "improvement": [
            "Добавить в промпт требование краткости и конкретности",
            "Использовать Multi-Query для лучшего понимания вопроса",
            "Реранкинг для выбора наиболее релевантных чанков",
        ]
    },
    "context_utilization": {
        "name": "Context Utilization",
        "desc": "Насколько извлечённый контекст соответствует вопросу",
        "improvement": [
            "Уменьшить chunk_size для более точных фрагментов",
            "Увеличить chunk_overlap для сохранения контекста",
            "Применить контекстное сжатие (EmbeddingsFilter)",
        ]
    },
}

for key, info in analysis.items():
    score = ragas_scores.get(key, 0)
    print(f"\n{info['name']} (score={score:.3f})")
    print(f"  {info['desc']}")
    if score < 0.8:
        print("  Рекомендуемые улучшения:")
        for tip in info["improvement"]:
            print(f"    - {tip}")
    else:
        print("  Качество отличное, улучшения не требуются")

# Сохраняем метрики в конфиг
config["ragas_scores"] = ragas_scores
with open("rag_config.json", "w", encoding="utf-8") as f:
    json.dump(config, f, ensure_ascii=False, indent=2)
print("\nМетрики сохранены в rag_config.json")

---
## 2. Оптимизация производительности

### 2.1 Базовое измерение времени ответа

In [ ]:
# Базовое измерение: 5 запросов без оптимизаций
benchmark_questions = [
    "Какая процентная ставка по потребительскому кредиту?",
    "Каков максимальный срок ипотеки?",
    "Условия досрочного погашения кредита",
    "Требования к заёмщику: возраст и доход",
    "Ставки по депозитам: максимальный доход",
]

print("Базовый бенчмарк (без оптимизаций)")

baseline_times = {"retrieval": [], "llm": [], "total": []}

for q in benchmark_questions:
    # Ретрив
    t0 = time.time()
    docs = base_retriever.invoke(q)
    t_retrieval = time.time() - t0

    # LLM
    context, sources = format_context(docs)
    messages = CHAT_PROMPT.format_messages(
        context=context, chat_history=[], question=q
    )
    t0 = time.time()
    _ = llm.invoke(messages)
    t_llm = time.time() - t0

    total = t_retrieval + t_llm
    baseline_times["retrieval"].append(t_retrieval)
    baseline_times["llm"].append(t_llm)
    baseline_times["total"].append(total)

    print(f"  Ретрив: {t_retrieval:.2f}с | LLM: {t_llm:.2f}с | Всего: {total:.2f}с  |  {q[:45]}")

print(f"  Среднее время ретрива : {sum(baseline_times['retrieval'])/len(baseline_times['retrieval']):.3f}с")
print(f"  Среднее время LLM     : {sum(baseline_times['llm'])/len(baseline_times['llm']):.3f}с")
print(f"  Среднее время ИТОГО   : {sum(baseline_times['total'])/len(baseline_times['total']):.3f}с")

baseline_avg = sum(baseline_times["total"]) / len(baseline_times["total"])

### 2.2 Оптимизация 1: Кеширование эмбеддингов запросов

In [ ]:
import hashlib

class CachedEmbeddingRetriever:
    """
    Ретривер с кешированием эмбеддингов запросов.
    Одинаковые запросы не пересчитываются повторно.
    """

    def __init__(self, vectorstore, k: int = 4):
        self.vectorstore = vectorstore
        self.k           = k
        self._cache      = {}
        self.hits        = 0
        self.misses      = 0

    def _query_hash(self, query: str) -> str:
        return hashlib.md5(query.lower().strip().encode()).hexdigest()

    def invoke(self, query: str):
        h = self._query_hash(query)

        if h in self._cache:
            self.hits += 1
            return self._cache[h]

        self.misses += 1
        docs = self.vectorstore.similarity_search(query, k=self.k)
        self._cache[h] = docs
        return docs

    def get_stats(self) -> dict:
        total    = self.hits + self.misses
        hit_rate = self.hits / total if total > 0 else 0
        return {
            "hits":     self.hits,
            "misses":   self.misses,
            "total":    total,
            "hit_rate": round(hit_rate, 3),
        }

    def clear(self):
        self._cache = {}
        self.hits = self.misses = 0


cached_retriever = CachedEmbeddingRetriever(vectorstore, k=4)
print("Кешированный ретривер создан")

# Тест кеша: первый прогон (cache miss) vs второй (cache hit)
test_query = "ставки по кредитам"

t0 = time.time()
_ = cached_retriever.invoke(test_query)
t_miss = time.time() - t0

t0 = time.time()
_ = cached_retriever.invoke(test_query)
t_hit = time.time() - t0

print(f"\nПервый вызов (cache miss): {t_miss:.4f}с")
print(f"Второй вызов (cache hit) : {t_hit:.4f}с")
speedup = t_miss / t_hit if t_hit > 0 else float('inf')
print(f"Ускорение                : x{speedup:.1f}")

### 2.3 Оптимизация 2: Батчинг LLM-запросов

In [ ]:
def rag_batch(questions: List[str]) -> List[dict]:
    """
    Пакетная обработка вопросов:
    1. Батч-ретрив всех вопросов сразу
    2. Батч-вызов LLM для всех сообщений
    """
    # 1. Ретрив для всех вопросов
    t0 = time.time()
    all_docs = [cached_retriever.invoke(q) for q in questions]
    t_retrieval = time.time() - t0

    # 2. Формируем все сообщения
    all_messages = []
    contexts_sources = []
    for q, docs in zip(questions, all_docs):
        context, sources = format_context(docs)
        contexts_sources.append(sources)
        msgs = CHAT_PROMPT.format_messages(
            context=context, chat_history=[], question=q
        )
        all_messages.append(msgs)

    # 3. Батч-вызов LLM (langchain поддерживает batch)
    t0 = time.time()
    responses = llm.batch(all_messages)
    t_llm = time.time() - t0

    results = []
    for q, resp, sources in zip(questions, responses, contexts_sources):
        answer = resp.content
        if sources and "[Источник:" not in answer:
            answer += f"\n\n[Источник: {', '.join(sources)}]"
        results.append({"question": q, "answer": answer})

    print(f"  Батч ретрив : {t_retrieval:.3f}с ({len(questions)} запросов)")
    print(f"  Батч LLM    : {t_llm:.3f}с ({len(questions)} запросов)")
    print(f"  Среднее на запрос: {(t_retrieval+t_llm)/len(questions):.3f}с")

    return results


# Тест батчинга
print("\nТест батчинга (5 запросов одновременно):")
batch_results = rag_batch(benchmark_questions)
for r in batch_results:
    print(f"  {r['question'][:50]}: {r['answer'][:80]}...")

### 2.4 Оптимизация 3: Сжатие контекста перед передачей в LLM

In [ ]:
from langchain.retrievers import ContextualCompressionRetriever
from langchain.retrievers.document_compressors import EmbeddingsFilter

# EmbeddingsFilter отбрасывает чанки с низким сходством с запросом
embeddings_filter = EmbeddingsFilter(
    embeddings=embeddings_model,
    similarity_threshold=0.75,
)

compression_retriever = ContextualCompressionRetriever(
    base_compressor=embeddings_filter,
    base_retriever=base_retriever,
)

# Сравнение размера контекста: без и с компрессией
test_q = "первоначальный взнос по семейной ипотеке"

base_docs       = base_retriever.invoke(test_q)
compressed_docs = compression_retriever.invoke(test_q)

base_tokens       = sum(len(d.page_content.split()) for d in base_docs)
compressed_tokens = sum(len(d.page_content.split()) for d in compressed_docs)

print(f"\nЗапрос: '{test_q}'")
print(f"  Без компрессии  : {len(base_docs)} чанков, ~{base_tokens} слов")
print(f"  С компрессией   : {len(compressed_docs)} чанков, ~{compressed_tokens} слов")
if base_tokens > 0:
    reduction = (1 - compressed_tokens / base_tokens) * 100
    print(f"  Сокращение      : {reduction:.0f}% (меньше токенов -> дешевле и быстрее)")

### 2.5 Итоговый бенчмарк: до и после оптимизаций

In [ ]:
print("ИТОГОВЫЙ БЕНЧМАРК: сравнение производительности")

# Оптимизированный вариант: кеш + компрессия
print("\nОптимизированный вариант (кеш + компрессия):")

# Прогрев кеша
_ = [cached_retriever.invoke(q) for q in benchmark_questions]

optimized_times = []
for q in benchmark_questions:
    t0   = time.time()
    docs = cached_retriever.invoke(q)           # из кеша
    context, sources = format_context(docs)
    messages = CHAT_PROMPT.format_messages(
        context=context, chat_history=[], question=q
    )
    _ = llm.invoke(messages)
    total = time.time() - t0
    optimized_times.append(total)
    print(f"  {total:.2f}с  |  {q[:50]}")

optimized_avg = sum(optimized_times) / len(optimized_times)

print("СРАВНЕНИЕ: ДО vs ПОСЛЕ оптимизации")
print(f"  Среднее время ДО  оптимизации : {baseline_avg:.3f}с")
print(f"  Среднее время ПОСЛЕ оптимизации: {optimized_avg:.3f}с")
speedup = baseline_avg / optimized_avg if optimized_avg > 0 else 0
print(f"  Ускорение                       : x{speedup:.2f}")
print()

cache_stats = cached_retriever.get_stats()
print(f"  Cache hits  : {cache_stats['hits']}")
print(f"  Cache misses: {cache_stats['misses']}")
print(f"  Hit rate    : {cache_stats['hit_rate']:.1%}")
print("=" * 65)

In [ ]:
# Визуализация результатов бенчмарка
import json

benchmark_summary = {
    "baseline": {
        "avg_retrieval_sec": round(sum(baseline_times["retrieval"])/len(baseline_times["retrieval"]), 3),
        "avg_llm_sec":       round(sum(baseline_times["llm"])/len(baseline_times["llm"]), 3),
        "avg_total_sec":     round(baseline_avg, 3),
    },
    "optimized": {
        "avg_total_sec":     round(optimized_avg, 3),
        "cache_hit_rate":    cache_stats["hit_rate"],
        "speedup_x":         round(speedup, 2),
    },
    "optimizations_applied": [
        "Кеширование эмбеддингов запросов (in-memory LRU)",
        "Контекстное сжатие (EmbeddingsFilter, threshold=0.75)",
        "Батчинг LLM-запросов (langchain .batch())",
    ]
}

print("Итоговый отчёт по производительности:")
print(json.dumps(benchmark_summary, ensure_ascii=False, indent=2))

# Сохраняем в конфиг
config["performance"] = benchmark_summary
with open("rag_config.json", "w", encoding="utf-8") as f:
    json.dump(config, f, ensure_ascii=False, indent=2)
print("\nОтчёт сохранён в rag_config.json")